# CMU-MOSEI Data Exploration

This notebook explores the CMU-MOSEI dataset and tests our multimodal model.

## Contents
1. Load dataset
2. Explore data statistics
3. Visualize samples
4. Test multimodal model
5. Extract attention weights

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.cmu_mosei_loader import CMUMOSEILoader, MOSEISample
from src.models.multimodal_model import SimpleMultimodalModel, MultimodalInferenceWrapper

%matplotlib inline
sns.set_style('whitegrid')

## 1. Load CMU-MOSEI Dataset

In [ ]:
# Initialize loader
loader = CMUMOSEILoader(data_dir="../data/cmu_mosei")

# Load small sample for exploration
print("Loading 50 samples...")
data = loader.load_aligned_features(split='train', max_samples=50, use_sdk=True)

print(f"Loaded {len(data['labels'])} samples")

## 2. Data Statistics

In [ ]:
# Dataset overview
print("=" * 60)
print("CMU-MOSEI Dataset Overview")
print("=" * 60)

print(f"\nNumber of samples: {len(data['labels'])}")
print(f"\nModalities:")
print(f"  Text shape: {data['text'][0].shape}")
print(f"  Audio shape: {data['audio'][0].shape}")
print(f"  Video shape: {data['video'][0].shape}")

# Sentiment distribution
labels = np.array(data['labels'])
print(f"\nSentiment Statistics:")
print(f"  Range: [{labels.min():.2f}, {labels.max():.2f}]")
print(f"  Mean: {labels.mean():.2f}")
print(f"  Std: {labels.std():.2f}")

# Label distribution
positive = (labels > 0.5).sum()
negative = (labels < -0.5).sum()
neutral = len(labels) - positive - negative

print(f"\nLabel Distribution:")
print(f"  Positive: {positive} ({positive/len(labels)*100:.1f}%)")
print(f"  Negative: {negative} ({negative/len(labels)*100:.1f}%)")
print(f"  Neutral: {neutral} ({neutral/len(labels)*100:.1f}%)")

In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(labels, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Sentiment Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Sentiment Score Distribution')
axes[0].axvline(x=0, color='red', linestyle='--', label='Neutral')
axes[0].legend()

# Box plot
axes[1].boxplot(labels, vert=True)
axes[1].set_ylabel('Sentiment Score')
axes[1].set_title('Sentiment Score Box Plot')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Explore Individual Samples

In [ ]:
# Get a few samples
print("Sample Utterances:\n")
print("=" * 80)

for i in range(min(5, len(data['labels']))):
    sample = loader.get_sample(data, i)
    
    print(f"\nSample {i+1}:")
    print(f"  ID: {sample.utterance_id}")
    print(f"  Text: {sample.text}")
    print(f"  Sentiment: {sample.sentiment_label} ({sample.sentiment_score:.2f})")
    print(f"  Speaker: {sample.speaker_id}")
    print("-" * 80)

## 4. Test Multimodal Model

In [ ]:
# Create model
print("Creating multimodal model...")
model = SimpleMultimodalModel(fusion_type='concat')
wrapper = MultimodalInferenceWrapper(model=model)

print(f"\nModel architecture:")
print(f"  Text encoder: BERT-base ({model.text_dim} dims)")
print(f"  Audio encoder: MLP (74 -> 256 dims)")
print(f"  Video encoder: MLP (35 -> 256 dims)")
print(f"  Fusion: {model.fusion_type}")
print(f"  Output: Sentiment regression")

In [ ]:
# Test on a few samples
print("Testing model on samples...\n")
print("=" * 80)

for i in range(min(3, len(data['labels']))):
    sample = loader.get_sample(data, i)
    
    # Predict (model is untrained, so predictions are random)
    result = wrapper.predict(
        text=sample.text,
        audio_features=sample.audio_features,
        video_features=sample.video_features,
        return_importance=True
    )
    
    print(f"\nSample {i+1}: {sample.text}")
    print(f"  True sentiment: {sample.sentiment_label} ({sample.sentiment_score:.2f})")
    print(f"  Predicted: {result['sentiment_label']} ({result['sentiment_score']:.2f})")
    print(f"  Modality importance:")
    for mod, imp in result['modality_importance'].items():
        print(f"    {mod}: {imp}%")
    print("-" * 80)

## 5. Visualize Modality Importance

In [ ]:
# Compute modality importance for multiple samples
importance_data = []

for i in range(min(10, len(data['labels']))):
    sample = loader.get_sample(data, i)
    result = wrapper.predict(
        sample.text,
        sample.audio_features,
        sample.video_features,
        return_importance=True
    )
    importance_data.append(result['modality_importance'])

# Create DataFrame
importance_df = pd.DataFrame(importance_data)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar chart
importance_df.plot(kind='bar', stacked=True, ax=axes[0], 
                   color=['steelblue', 'coral', 'lightgreen'])
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Importance (%)')
axes[0].set_title('Modality Importance per Sample (Stacked)')
axes[0].legend(title='Modality')
axes[0].set_ylim([0, 100])

# Average importance
avg_importance = importance_df.mean()
avg_importance.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral', 'lightgreen'])
axes[1].set_xlabel('Modality')
axes[1].set_ylabel('Average Importance (%)')
axes[1].set_title('Average Modality Importance')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("\nAverage Modality Importance:")
for mod, imp in avg_importance.items():
    print(f"  {mod}: {imp:.2f}%")

## 6. Next Steps

**Note**: The model predictions are currently random because the model is untrained.

To get meaningful results:
1. Train the model on CMU-MOSEI (or load pre-trained weights)
2. Fine-tune on your specific task
3. Evaluate on test set

For this minimal implementation, we'll focus on using the **pre-extracted features** and **attention weights** for explanation, rather than training from scratch.

In [ ]:
# Save exploration results
summary = {
    'n_samples': len(data['labels']),
    'sentiment_range': [float(labels.min()), float(labels.max())],
    'sentiment_mean': float(labels.mean()),
    'positive_pct': float(positive / len(labels) * 100),
    'negative_pct': float(negative / len(labels) * 100),
    'neutral_pct': float(neutral / len(labels) * 100),
    'avg_modality_importance': avg_importance.to_dict()
}

print("\nData Exploration Summary:")
import json
print(json.dumps(summary, indent=2))